# Sprint 3 Live — Streaming and observability workshops

Use this one Live notebook for LS09–LS12 in a separate Live workspace. Keep Campus assessment work in its own notebook and export.

The route forwards real OpenRouter output. No scripted text is used as model output. Each admitted supported request can use provider quota. The original Module A evaluator remains a separate deterministic check of evidence, tool, and decision-flag behavior.

**Colab runtime:** `ServiceProcess` runs Uvicorn in a managed child process bound to loopback, checks readiness with a timeout, and provides an explicit stop. Do not run `uvicorn.run()` in a cell or expose the service through ngrok; a foreground server call would prevent that cell from returning. The streaming client reads until a terminal event or its bounded timeout, then the notebook can continue to logging and evaluation.


## 1. Set up once

In [ ]:
import json
import os
from pathlib import Path
import runpy
import secrets
import shutil
import subprocess
import sys
from zipfile import ZipFile

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path("/content/ai-app-dev_Mod-B")
PROJECT = Path("/content/fieldcare-sprint-3-live")
CAMPUS_CHECKPOINT = ""  # Optional uploaded secured Sprint 2 source checkpoint.
LOG_PATH = PROJECT / "logs" / "requests.jsonl"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
else:
    # Colab runtimes persist /content; refresh an existing clone before reinstalling it.
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))  # Make the editable package visible to this running kernel.

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)

if CAMPUS_CHECKPOINT:
    PROJECT.mkdir(parents=True, exist_ok=True)
    with ZipFile(CAMPUS_CHECKPOINT) as archive:
        archive.extractall(PROJECT)
elif not PROJECT.exists():
    shutil.copytree(REPO / "examples" / "fieldcare", PROJECT)

settings_path = PROJECT / "app" / "security_settings.py"
if not settings_path.exists():
    settings_path.write_text("""from module_b.security import LimitPolicy
CALLER_ENV = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
POLICY = LimitPolicy(allowance=2, window_seconds=60)
""")
settings = runpy.run_path(str(settings_path))
CALLERS = settings["CALLER_ENV"]
caller_keys = {label: secrets.token_urlsafe(32) for label in CALLERS}
for label, variable in CALLERS.items():
    os.environ[variable] = caller_keys[label]

print("Module B source:", REPO)
print("Sprint 3 workspace:", PROJECT)
print("Starting point:", "Sprint 2 checkpoint" if CAMPUS_CHECKPOINT else "labelled recovery workspace")

## LS09 — Stream real provider output

Inspect the route in `app/stream_routes.py`, then run the stream request. Follow each NDJSON event as the client receives it. For a supported request, the route first sends `metadata` (status, citation IDs, mode, and request ID), then real provider `delta` events, optional usage, and a terminal `complete` or safe `error` event. A valid missing-equipment request takes the deterministic JSON clarification path instead. Identify the request ID, event order, terminal state, and actual text.


In [ ]:
from module_b.observability import FIELDS, ObservationMiddleware
from module_b.runtime import ServiceProcess
import httpx
import time

stream_source = REPO / "examples" / "patterns" / "sprint_3" / "stream_routes.py"
stream_file = PROJECT / "app" / "stream_routes.py"
if not stream_file.exists():
    shutil.copyfile(stream_source, stream_file)

main_path = PROJECT / "app" / "main.py"
main_source = main_path.read_text()
route_registration = """
from app.stream_routes import router as stream_router
app.include_router(stream_router)
"""
security_marker = "from module_b.security import SecurityMiddleware"
if "diagnose_stream" not in main_source:
    if security_marker in main_source:
        main_source = main_source.replace(security_marker, route_registration + "\n" + security_marker, 1)
    else:
        main_source += route_registration + """
from module_b.security import SecurityMiddleware, protected_post_paths
from app.security_settings import CALLER_ENV, POLICY
app.add_middleware(SecurityMiddleware, caller_env=CALLER_ENV,
                   protected_paths=protected_post_paths(app), policy=POLICY)
"""
main_source += f"""
from module_b.observability import ObservationMiddleware
app.add_middleware(
    ObservationMiddleware,
    path={str(LOG_PATH)!r},
    routes=("/v1/diagnose", "/v1/diagnose-stream"),
    approved_models=(os.environ["OPENROUTER_MODEL"],),
)
"""
if "import os" not in main_source:
    main_source = "import os\n" + main_source
main_path.write_text(main_source)

def start_service():
    service = ServiceProcess(
        project_dir=PROJECT,
        env={variable: caller_keys[label] for label, variable in CALLERS.items()},
    )
    service.start()
    return service

service = start_service()
labels = list(CALLERS)
stream_caller = labels[0]
buffered_caller = labels[1] if len(labels) > 1 else labels[0]
payload = {
    "question": "Which filter and airflow checks are documented?",
    "equipment_id": "EQ-FC-1002",
}

In [ ]:
events = []
arrival_seconds = []
started = time.perf_counter()
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    with client.stream(
        "POST",
        "/v1/diagnose-stream",
        json=payload,
        headers={"X-API-Key": caller_keys[stream_caller]},
    ) as response:
        request_id = response.headers.get("x-request-id")
        print("HTTP:", response.status_code, "request:", request_id)
        for line in response.iter_lines():
            if not line:
                continue
            event = json.loads(line)
            events.append(event)
            arrival_seconds.append(time.perf_counter() - started)
            print(event)

text = "".join(event.get("text", "") for event in events if event.get("type") == "delta")
terminal = next((event for event in reversed(events)
                 if event.get("type") in {"complete", "error"}), None)
usage = next((event for event in events if event.get("type") == "usage"), None)
print("Generated text:", text)
print("Terminal event:", terminal)
print("Provider usage:", usage.get("tokens") if usage else None,
      "(unknown when absent)")

## LS10 — Inspect safe request signals

Read the actual records for the streamed and buffered requests. Which fields identify an attempt, model, usage, latency, outcome, and error? Which fields are absent because the service did not retain prompts or authorization headers?

In [ ]:
records = [json.loads(line) for line in LOG_PATH.read_text().splitlines() if line]
for record in records:
    print({field: record.get(field) for field in FIELDS})
print("Request IDs:", [row["request_id"] for row in records])
print("Prompt retained:", any("prompt" in row for row in records))
print("Authorization header retained:", any("headers" in row for row in records))

## LS10 — Apply the safe-field rule

Use only the synthetic marker below. Decide what the service needs for debugging, then compare that decision with the allowlisted record.

In [ ]:
from module_b.observability import safe_record

marker = "SYNTHETIC-NOT-A-CREDENTIAL"
observed = next(record for record in records if record["request_id"] == request_id)
candidate = {
    **observed,
    "_approved_models": (os.environ["OPENROUTER_MODEL"],),
    "prompt": "Synthetic private marker " + marker,
    "headers": {"X-API-Key": marker},
}
minimized = safe_record(
    candidate,
    routes=("/v1/diagnose", "/v1/diagnose-stream"),
)
assert minimized["request_id"] == request_id
assert minimized["model"] == observed["model"]
assert marker not in repr(minimized)
print("Based on actual request:", request_id)
print("Retained fields:", minimized)
print("Marker in retained record:", marker in repr(minimized))

## LS11 — Feed a signal into the existing evaluation

Choose the original evaluation cases related to the signal you observed. Run the original deterministic evaluator on the saved design. Explain what its document/tool/flag checks establish and what they do not say about model prose or stream delivery.

In [ ]:
from module_b._module_a import project as original
from module_b.evaluation import load_pipeline, evaluate_selected

design_path = PROJECT / "data" / "pipeline_design.json"
design_path.parent.mkdir(parents=True, exist_ok=True)
if not design_path.exists():
    design_path.write_text(json.dumps(original.default_pipeline_design(), indent=2) + "\\n")

pipeline = load_pipeline(design_path)
case_ids = ["EVAL-FC-005", "EVAL-FC-003"]
report = evaluate_selected(design_path, case_ids)
for row in report["rows"]:
    print({
        "eval_id": row["eval_id"],
        "pipeline_pass": row["pipeline_pass"],
        "missing_required_docs": row["missing_required_docs"],
        "missing_required_tools": row["missing_required_tools"],
        "missing_expected_flags": row["missing_expected_flags"],
    })
print("Design revision:", report["design_revision"])
print("Evaluation behavior is deterministic application evidence; it does not score generated prose.")

## LS12 — Demonstrate the evidence chain

Show one real stream request, its correlated safe record, the same service’s buffered contract, and a related original evaluation result. State one supported conclusion and one limitation. Export the Live workspace and save this notebook separately.

In [ ]:
service.stop()
from zipfile import ZIP_DEFLATED, ZipFile

CHECKPOINT_OUT = Path("/content/sprint-3-live-checkpoint.zip")
with ZipFile(CHECKPOINT_OUT, "w", ZIP_DEFLATED) as archive:
    for folder in ("app", "data", "logs"):
        path = PROJECT / folder
        if not path.exists():
            continue
        for item in path.rglob("*"):
            if item.is_file() and "__pycache__" not in item.parts and item.suffix != ".pyc":
                archive.write(item, item.relative_to(PROJECT))
print("Saved checkpoint:", CHECKPOINT_OUT)

### Carry forward

Keep this Live workspace separate from Campus. Carry the secured, observable service checkpoint you intend to extend, with the source revision and evidence from the same run.